In [12]:
import pandas as pd 
subs=pd.read_csv(r'../data/subscriptions.csv')
rev=pd.read_csv(r'../data/monthly_revenue.csv')

print("==Shapes==")
print(f'Subscriptions:{subs.shape}')

print(f"Monthly Revenue:{rev.shape}")

print("==Subscriptions Columns==")
print(subs.columns.tolist())

print("Monthly Revenue Columns==")
print(rev.columns.tolist())

print("==Subscriptions Head==")
print(subs.head(3))


print("==Revenue Head==")
print(rev.head(3))

==Shapes==
Subscriptions:(600, 17)
Monthly Revenue:(48, 8)
==Subscriptions Columns==
['customer_id', 'plan', 'billing_cycle', 'industry', 'company_size', 'seats', 'monthly_revenue', 'acquisition_channel', 'region', 'signup_date', 'churned', 'churn_date', 'churn_reason', 'support_tickets_12mo', 'nps_score', 'feature_usage_pct', 'upgraded']
Monthly Revenue Columns==
['month', 'total_active_customers', 'new_customers', 'churned_customers', 'monthly_churn_rate_pct', 'total_mrr', 'avg_revenue_per_customer', 'customer_acquisition_cost']
==Subscriptions Head==
  customer_id          plan billing_cycle industry company_size  seats  \
0   CUST-0001  Professional       Monthly   Retail       51-200     64   
1   CUST-0002  Professional        Annual   Retail       51-200     35   
2   CUST-0003       Starter        Annual  Finance         500+    266   

   monthly_revenue acquisition_channel         region signup_date churned  \
0           825.55      Organic Search  North America  2022-06-27 

In [13]:
print('== Subscriptions Nulls==')
print(subs.isnull().sum())

print("\n==revenue Nulls==")
print(rev.isnull().sum())

print("==Duplicates==")
print(f"Subscriptions:{subs.duplicated().sum()}")
print(f"Monthly Revenue :{rev.duplicated().sum()}")

print("==Subscriptions dataTypes==")
print(subs.dtypes)

print("\n revenue DataTypes==")
print(rev.dtypes)


print("==Unique values==")
print("Plans:",subs['plan'].unique())
print("Billing cycle:",subs['billing_cycle'].unique())
print("Churned:",subs['churned'].unique())
print("Churn Reasons:",subs['churn_reason'].unique())
print("Company Sizes:",subs['company_size'].unique())
print("Industries:",subs['industry'].unique())
print("Acq channels:",subs['acquisition_channel'].unique())
print("Reigons:",subs['region'].unique())
print("Upgraded:",subs['upgraded'].unique())

# ── Key stats ──
print("=== SUBSCRIPTION STATS ===")
print(subs[['seats', 'monthly_revenue', 'support_tickets_12mo',
            'nps_score', 'feature_usage_pct']].describe().round(2))

print("\n=== REVENUE STATS ===")
print(rev[['total_mrr', 'monthly_churn_rate_pct','customer_acquisition_cost','avg_revenue_per_customer']].describe().round(2))


print("==Churn BreakDown==")
churn_counts=subs['churned'].value_counts()
print(churn_counts)
print(f"\n Overall Churn Rate:{churn_counts['Yes']/len(subs)*100:.2f}%")

print("\n== Churn By Plan==")
print(subs.groupby('plan')['churned'].value_counts(normalize=True).round(3)*100)


print("==AVG NPS:Churned vs Active==")
print(subs.groupby('churned')[['nps_score','feature_usage_pct','support_tickets_12mo']].mean().round(2))

== Subscriptions Nulls==
customer_id               0
plan                      0
billing_cycle             0
industry                  0
company_size              0
seats                     0
monthly_revenue           0
acquisition_channel       0
region                    0
signup_date               0
churned                   0
churn_date              287
churn_reason            287
support_tickets_12mo      0
nps_score                 0
feature_usage_pct         0
upgraded                  0
dtype: int64

==revenue Nulls==
month                        0
total_active_customers       0
new_customers                0
churned_customers            0
monthly_churn_rate_pct       0
total_mrr                    0
avg_revenue_per_customer     0
customer_acquisition_cost    0
dtype: int64
==Duplicates==
Subscriptions:0
Monthly Revenue :0
==Subscriptions dataTypes==
customer_id                 str
plan                        str
billing_cycle               str
industry                    str


## Data Quality Summary — CloudTask Pro

| Check | Subscriptions | Monthly Revenue |
|---|---|---|
| Null values | churn_date/reason nulls = active customers ✅ | 0 ✅ |
| Duplicates | 0 ✅ | 0 ✅ |
| Type fixes | signup_date, churn_date → datetime ✅ | month → datetime ✅ |
| Outliers | None requiring removal ✅ | — |

## Key Pre-Analysis Findings

- **Overall churn rate: 52.17%** — 313 of 600 customers churned
- **Starter plan: 70.5% churn** — highest risk segment
- **Enterprise plan: 22% churn** — most stable segment
- **Churned customers use only 27.45% of features** vs 55% for active
- **Churned customers NPS: 3.04** vs 5.81 for active
- **Feature usage below 30% = at-risk threshold**

## New Columns Added
- `churned_binary` — 1 for churned, 0 for active
- `lifespan_months` — months from signup to churn/today
- `clv` — monthly_revenue × lifespan_months
- `nps_category` — Detractor/Passive/Promoter
- `at_risk` — 1 if feature_usage_pct < 30%
- `upgraded_binary` — 1 if upgraded, 0 if not

In [14]:
subs['signup_date']=pd.to_datetime(subs['signup_date'])
subs['churn_date']=pd.to_datetime(subs['churn_date'])
rev['month']=pd.to_datetime(rev['month'])

subs['churn_reason']=subs['churn_reason'].fillna('Active')

subs['churned_binary']=(subs['churned']=='Yes').astype(int)

refrence_date=pd.Timestamp('2025-12-31')

subs['lifespan_months']=subs.apply(
    lambda x:(x['churn_date']-x['signup_date']).days/30.44
    if pd.notna(x['churn_date'])
    else (refrence_date-x['signup_date']).days/30.44,
    axis=1
).round(1)

# Clv- Customer LifeTime Value--monthly_rev* lifespan_months
subs['clv']=(subs['monthly_revenue']*subs['lifespan_months']).round(2)

subs['upgraded_binary']=(subs['upgraded']=='Yes').astype(int)

subs['nps_category']=pd.cut(
    subs['nps_score'],
    bins=[0,3,6,10],
    labels=['Detractor','Passive','Promoter']
)

# Adding a Risk flag
subs['at_risk']=(subs['feature_usage_pct']<30).astype(int)


size_order={'1-10':1,'11-50':2,'51-200':3,'201-500':4,'500+':5}
subs['company_size_order']=subs['company_size'].map(size_order)

print("=== CLEANING SUMMARY ===")
print(f"Total customers    : {len(subs)}")
print(f"Churned            : {subs['churned_binary'].sum()}")
print(f"Active             : {(subs['churned_binary'] == 0).sum()}")
print(f"\nNew columns added  : lifespan_months, clv, churned_binary,")
print(f"                     upgraded_binary, nps_category, at_risk")
print(f"\nAvg lifespan (churned) : {subs[subs['churned']=='Yes']['lifespan_months'].mean():.1f} months")
print(f"Avg lifespan (active)  : {subs[subs['churned']=='No']['lifespan_months'].mean():.1f} months")
print(f"\nAvg CLV (churned)  : ${subs[subs['churned']=='Yes']['clv'].mean():,.2f}")
print(f"Avg CLV (active)   : ${subs[subs['churned']=='No']['clv'].mean():,.2f}")
print(f"\nAt-risk customers  : {subs['at_risk'].sum()} ({subs['at_risk'].mean()*100:.1f}%)")



=== CLEANING SUMMARY ===
Total customers    : 600
Churned            : 313
Active             : 287

New columns added  : lifespan_months, clv, churned_binary,
                     upgraded_binary, nps_category, at_risk

Avg lifespan (churned) : 9.7 months
Avg lifespan (active)  : 17.8 months

Avg CLV (churned)  : $8,139.54
Avg CLV (active)   : $20,850.65

At-risk customers  : 223 (37.2%)


In [15]:

subs.to_csv('../data/subscriptions_clean.csv', index=False)
rev.to_csv('../data/monthly_revenue_clean.csv', index=False)

print("Clean files exported")
print(f"subscriptions_clean  : {len(subs)} rows, {len(subs.columns)} cols")
print(f"monthly_revenue_clean: {len(rev)} rows, {len(rev.columns)} cols")

Clean files exported
subscriptions_clean  : 600 rows, 24 cols
monthly_revenue_clean: 48 rows, 8 cols
